In [10]:
import os
import sys
from pathlib import Path

# If auto-detect fails, set this manually to your repo root.
REPO_ROOT = Path('C:/Users/Thomas/.0 Thesis')

def _is_repo_root(path: Path) -> bool:
    return (path / 'training_config.json').exists() and (path / 'env').exists()

def _find_repo_root(start: Path) -> Path | None:
    cur = start
    while True:
        if _is_repo_root(cur):
            return cur
        if cur.parent == cur:
            return None
        cur = cur.parent

def _shallow_search(base: Path, max_depth: int = 3) -> Path | None:
    if not base.exists():
        return None
    base = base.resolve()
    for root, dirs, files in os.walk(base):
        depth = len(Path(root).relative_to(base).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        root_path = Path(root)
        if _is_repo_root(root_path):
            return root_path
    return None

repo_root = REPO_ROOT
if repo_root is None:
    repo_root = _find_repo_root(Path.cwd())
if repo_root is None:
    # Colab default workdir is often /content
    repo_root = _shallow_search(Path('/content'))

if repo_root is None:
    raise RuntimeError('Could not find repo root. Set REPO_ROOT manually.')

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', os.getcwd())


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Thomas/.0 Thesis'

In [11]:
import os
from pathlib import Path

# list top-level dirs
print(os.listdir("/content"))

# search for a folder that contains training_config.json and env/
def find_repo_root(base="/content", max_depth=4):
    base = Path(base)
    for root, dirs, files in os.walk(base):
        depth = len(Path(root).relative_to(base).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        root_path = Path(root)
        if (root_path / "training_config.json").exists() and (root_path / "env").exists():
            return root_path
    return None

print("Repo root:", find_repo_root("/content"))


['.config', 'sample_data']
Repo root: None


# PortEnv Training Readiness Check

Fail-fast sanity checks to ensure the environment is stable and safe for training.

In [ ]:
import numpy as np

from env.port_env import PortEnv
from env.port_env_spec import encode_action, decode_action

SEED = 42
ROLLOUT_STEPS = 50

def assert_true(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(message)

env = PortEnv()
obs, info = env.reset(seed=SEED)

# Spaces & shapes
assert_true(obs.shape == (37,), f'Expected obs shape (37,), got {obs.shape}')
assert_true(obs.dtype == np.float32, f'Expected obs dtype float32, got {obs.dtype}')
assert_true(np.all(obs >= 0.0) and np.all(obs <= 1.0), 'Observation values must be in [0, 1]')
assert_true(env.action_space.n > 0, 'Action space must be non-empty')

# Reset/step contract
step_out = env.step(env.no_op_action)
assert_true(len(step_out) == 5, f'Expected 5 outputs from step, got {len(step_out)}')
obs2, reward, terminated, truncated, info2 = step_out
assert_true(obs2.shape == (37,), f'Expected obs shape (37,), got {obs2.shape}')

# Encode/decode sanity
noop = encode_action(env, vessel_slot=env.no_op_slot, quay_position=0, cranes=0)
decoded = decode_action(env, noop)
assert_true(decoded == (env.no_op_slot, 0, 0), f'No-op decode mismatch: {decoded}')

# Invariants
assert_true(0 <= env.cranes_in_use <= env.total_cranes_limit, 'cranes_in_use out of bounds')
assert_true(np.all((env.quay_map == 0) | (env.quay_map == 1)), 'quay_map must be 0/1')
current_step_before = env.current_step
env.step(env.no_op_action)
assert_true(env.current_step == current_step_before + 1, 'current_step must increment by 1 per step')

# Random rollout
env.reset(seed=SEED)
total_reward = 0.0
terminated = False
truncated = False
for _ in range(ROLLOUT_STEPS):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    assert_true(obs.shape == (37,), f'Expected obs shape (37,), got {obs.shape}')
    assert_true(np.all(obs >= 0.0) and np.all(obs <= 1.0), 'Observation values must be in [0, 1]')
    assert_true(0 <= env.cranes_in_use <= env.total_cranes_limit, 'cranes_in_use out of bounds')
    if terminated or truncated:
        break

print('PASS: PortEnv readiness checks succeeded')
print(f'Random rollout: steps={_ + 1}, total_reward={total_reward:.3f}, terminated={terminated}, truncated={truncated}')
